In [16]:
import os
import warnings
import joblib
import folium
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import pearsonr
import tensorflow as tf

# =========================
# CONFIGURATION
# =========================
FILE_NAME = "mother_teresa_uni"   # without .csv

BASE_FOLDER = r"../../../../data/raw/binary_mannual_collection"
CSV_PATH = os.path.join(BASE_FOLDER, FILE_NAME + ".csv")

SCALER_CNN_PATH = r"../scaler_cnn.pkl"
SCALER_MLP_PATH = r"../scaler_mlp.pkl"
TFLITE_MODEL_PATH = r"../pothole_model.tflite"

os.makedirs("sensor_plots", exist_ok=True)
os.makedirs("map_plots", exist_ok=True)

# =========================
# Load data
# =========================
df = pd.read_csv(CSV_PATH)

# =========================
# FIX: Pause/Resume using real datetime column
# =========================
df['time'] = pd.to_datetime(df['time'], format='ISO8601', utc=True)

# seconds between rows
df['dt'] = df['time'].diff().dt.total_seconds().fillna(0)

# if gap > 1 sec → new segment
GAP_THRESHOLD = 1.0
df['segment'] = (df['dt'] > GAP_THRESHOLD).cumsum()

# =========================
# Create SAFE 20-row windows
# =========================
WINDOW = 20
window_ids = []
current_id = 0

for _, seg in df.groupby('segment'):
    n = len(seg)
    ids = np.arange(n) // WINDOW + current_id
    window_ids.extend(ids)
    if len(ids) > 0:
        current_id = ids.max() + 1

df['window_id'] = window_ids
df = df.groupby('window_id').filter(lambda x: len(x) == WINDOW)

feature_cols = ['ax', 'ay', 'az', 'wx', 'wy', 'wz', 'speed']

# =========================
# Feature extraction
# =========================
def extract_stat_features(series, stats_row):
    series = np.nan_to_num(series)
    stats_row.append(np.mean(series))
    stats_row.append(np.std(series))
    stats_row.append(np.sqrt(np.mean(series**2)))
    stats_row.append(np.max(series) - np.min(series))

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        stats_row.append(stats.skew(series, bias=False))
        stats_row.append(stats.kurtosis(series, fisher=True, bias=False))

groups = [g for _, g in df.groupby('window_id')]

X_cnn_raw = np.array([g[feature_cols].values for g in groups])
X_mlp_raw = []

for g in groups:
    stats_row = []
    wx, wy = g['wx'].values, g['wy'].values
    az, ay = g['az'].values, g['ay'].values

    cov = pearsonr(wx, wy)[0] if (np.std(wx) > 1e-5 and np.std(wy) > 1e-5) else 0
    roll_ratio = np.std(wy) / (np.std(wx) + 1e-6)
    slope = np.mean(np.gradient(wx[:10]))

    stats_row.extend([
        cov,
        roll_ratio,
        slope,
        np.min(az),
        np.ptp(az),
        np.mean(ay),
        np.mean(g['speed'])
    ])

    for col in feature_cols:
        extract_stat_features(g[col].values, stats_row)

    X_mlp_raw.append(stats_row)

X_mlp_raw = np.array(X_mlp_raw)

# =========================
# Scaling
# =========================
sc_cnn = joblib.load(SCALER_CNN_PATH)
sc_mlp = joblib.load(SCALER_MLP_PATH)

N, T, F = X_cnn_raw.shape
X_cnn = sc_cnn.transform(X_cnn_raw.reshape(-1, F)).reshape(N, T, F)
X_mlp = sc_mlp.transform(X_mlp_raw)

# =========================
# TFLite inference
# =========================
interpreter = tf.lite.Interpreter(model_path=TFLITE_MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

predictions = []

for i in range(N):
    interpreter.set_tensor(input_details[0]['index'], X_cnn[i:i+1].astype(np.float32))
    interpreter.set_tensor(input_details[1]['index'], X_mlp[i:i+1].astype(np.float32))
    interpreter.invoke()

    output = interpreter.get_tensor(output_details[0]['index'])
    pred = np.argmax(output)
    predictions.append(pred)

df['pred'] = np.repeat(predictions, WINDOW)

# =========================
# UPDATED LABEL COLORS (4 Classes)
# 0: Good, 1: Vibration, 2: Pothole, 3: Big Pothole
# =========================
label_colors = {
    0: "#00FF00",   # green
    1: "#FFA500",   # orange
    2: "#FF0000",   # red
    3: "#FF00FF"    # magenta
}

# =========================
# Sensor Plot
# =========================
features = feature_cols
x_full = np.arange(len(df))

plt.figure(figsize=(18, 12))

for idx, feature in enumerate(features, 1):
    plt.subplot(len(features), 1, idx)
    start = 0
    while start < len(df):
        current_label = df['pred'].iloc[start]
        end = start
        while end < len(df) and df['pred'].iloc[end] == current_label:
            end += 1
        
        # Plot segment with color matching the prediction
        plt.plot(
            x_full[start:end],
            df[feature].iloc[start:end],
            color=label_colors.get(current_label, "#808080"), # Default to gray if unknown
            linewidth=1
        )
        start = end

    plt.title(feature)
    plt.grid(True)

plt.tight_layout()
plt.savefig(f"sensor_plots/{FILE_NAME}.png")
plt.close()

# =========================
# Folium Map
# =========================
df = df.dropna(subset=['latitude', 'longitude']).copy()

if not df.empty:
    center = [df['latitude'].mean(), df['longitude'].mean()]
    m = folium.Map(location=center, zoom_start=17)

    for _, row in df.iterrows():
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=3,
            color=label_colors.get(row['pred'], "#808080"),
            fill=True,
            fill_color=label_colors.get(row['pred'], "#808080"),
            fill_opacity=1,
            weight=0
        ).add_to(m)

    m.save(f"map_plots/{FILE_NAME}.html")
    print(f"Inference complete. Plots saved for {FILE_NAME}.")
else:
    print("No GPS data found to plot map.")

c:\Users\nishk\anaconda3\envs\py310\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Inference complete. Plots saved for mother_teresa_uni.
